# Scriptorium — Barton HTR Training (Colab)

Fine-tunes a Kraken HTR model on the 3,148 labelled row crops from the 1850 U.S. Census (Barton, Tioga Co., NY).

**How to use:** Runtime → Change runtime type → set Hardware accelerator to **T4 GPU**. Then Runtime → Run all.

The trained model is copied to `/content/drive/MyDrive/scriptorium/barton_htr.mlmodel` at the end (requires Drive mount when prompted).

In [ ]:
# 1. Install kraken + git-lfs (both usually preinstalled on Colab but ensure).
!apt-get -qq install -y git-lfs > /dev/null
!git lfs install --skip-repo > /dev/null
!pip install -q kraken
!kraken --version

In [ ]:
# 2. Clone the repo (LFS pointers only for now).
%cd /content
!rm -rf scriptorium-rails
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/kraftinator/scriptorium-rails.git
%cd scriptorium-rails
!git lfs pull -I data/barton_htr_data.tar.gz
!ls -lh data/barton_htr_data.tar.gz

In [ ]:
# 3. Extract training data and emit .gt.txt companions.
%cd /content/scriptorium-rails
!mkdir -p data-extract
!tar xzf data/barton_htr_data.tar.gz -C data-extract
# Rewrite manifest paths to point at the extracted location, then emit .gt.txt files.
!sed -i 's|/Users/admin/scriptorium/data|/content/scriptorium-rails/data-extract|g' data-extract/manifest.jsonl
!python python/prep_kraken_gt.py data-extract/manifest.jsonl
# Build the train_images.lst.
!find data-extract/crops -name 'line_*.png' | while read p; do [ -f "${p%.png}.gt.txt" ] && echo "$p"; done > data-extract/train_images.lst
!wc -l data-extract/train_images.lst

In [ ]:
# 4. Verify GPU is available.
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 5. Train. Batch 32 on GPU; early stopping with patience 5.
!mkdir -p models
!ketos -v train \
    -B 32 \
    -o models/barton_htr \
    -q early \
    --lag 5 \
    -F 1.0 \
    -d cuda:0 \
    -f path \
    --workers 4 \
    -t data-extract/train_images.lst 2>&1 | tail -100

In [ ]:
# 6. Save the best model to Drive so it survives after the runtime disconnects.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/scriptorium
!ls -lh models/
# Pick the highest-numbered checkpoint as the best (Kraken saves per-epoch);
# copy it into Drive with a stable filename.
!cp "$(ls -t models/barton_htr_*.mlmodel | head -1)" /content/drive/MyDrive/scriptorium/barton_htr.mlmodel
!ls -lh /content/drive/MyDrive/scriptorium/